# Лабораторная №10 — генерация лиц с помощью GAN (CelebA)

Все восемь этапов методички: 1. загрузка CelebA; 2. дискриминатор; 3. генератор; 4. состязательная модель; 5. обучение дискриминатора; 6. обучение генератора; 7. общий цикл; 8. результаты.

**Важное исправление методички:** данные загружаются как RGB 64×64×3, хотя текст описывает выход 28×28×1. Здесь генератор и дискриминатор согласованы с реальным форматом **64×64×3**. Для Colab рекомендуется GPU.

## 1. Загрузка CelebA

Сначала пробуем ссылку Google Drive из PDF. Если доступ ограничен, загрузите архив CelebA вручную и укажите `ARCHIVE_PATH`. Не используем случайные изображения вместо CelebA.

In [ ]:
%pip -q install gdown
import os, pathlib, zipfile, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(42)
os.makedirs("celeba_gan",exist_ok=True)
ARCHIVE_PATH="celeba_gan/data.zip"
if not pathlib.Path(ARCHIVE_PATH).exists():
    import gdown
    gdown.download(id="1O7m1010EJjLE5QxLZiM9Fpjs7Oj6e684",output=ARCHIVE_PATH,quiet=False)
assert pathlib.Path(ARCHIVE_PATH).exists(),"Ссылка не сработала: скачайте CelebA вручную в celeba_gan/data.zip"
with zipfile.ZipFile(ARCHIVE_PATH) as z:
    z.extractall("celeba_gan")
print("Архив распакован")

### 1.1. Создание датасета изображений без меток и визуализация

Используем tf.data с ресайзом до 64×64, нормализацией [0,1] и небольшим подмножеством для быстрого учебного прогона. При необходимости увеличьте MAX_IMAGES.

In [ ]:
paths=sorted(str(p) for p in pathlib.Path("celeba_gan").rglob("*") if p.suffix.lower() in (".jpg",".jpeg",".png"))
assert paths,"После распаковки не найдены фотографии"
MAX_IMAGES=5000
paths=paths[:MAX_IMAGES]
def decode(path):
    img=tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
    img.set_shape([None,None,3])
    img=tf.image.resize_with_crop_or_pad(img,tf.maximum(tf.shape(img)[0],tf.shape(img)[1]),tf.maximum(tf.shape(img)[0],tf.shape(img)[1]))
    img=tf.image.resize(img,(64,64))
    return tf.cast(img,tf.float32)/255.
dataset=tf.data.Dataset.from_tensor_slices(paths).shuffle(len(paths),seed=42).map(decode,num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)
sample=next(iter(dataset))
fig,axs=plt.subplots(2,4,figsize=(8,4))
for i,ax in enumerate(axs.flat):
    ax.imshow(sample[i]);ax.axis("off")
plt.tight_layout();plt.show()

## 2. Дискриминатор — допишите код

Conv2D с понижением разрешения, LeakyReLU, Dropout и выходной Dense(1) с логитом.

In [ ]:
discriminator=keras.Sequential([
    keras.Input(shape=(64,64,3)),
    layers.Conv2D(64,4,strides=2,padding="same"),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Dropout(0.3),
    layers.Conv2D(128,4,strides=2,padding="same"),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Dropout(0.3),
    layers.Conv2D(256,4,strides=2,padding="same"),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Flatten(),
    layers.Dense(1)
],name="discriminator")
discriminator.summary()

## 3. Генератор — допишите код

128-мерный латентный вектор преобразуется в RGB-изображение 64×64 через Dense/Reshape/Conv2DTranspose.

In [ ]:
latent_dim=128
generator=keras.Sequential([
    keras.Input(shape=(latent_dim,)),
    layers.Dense(8*8*256),
    layers.Reshape((8,8,256)),
    layers.Conv2DTranspose(128,4,strides=2,padding="same"),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Conv2DTranspose(64,4,strides=2,padding="same"),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Conv2DTranspose(32,4,strides=2,padding="same"),
    layers.LeakyReLU(negative_slope=0.2),
    layers.Conv2D(3,3,padding="same",activation="sigmoid")
],name="generator")
generator.summary()
print("Форма выхода:",generator(tf.random.normal((1,latent_dim))).shape)

## 4. Генеративно-состязательная сеть

Используем кастомный цикл обучения GAN с GradientTape, чтобы корректно обновлять веса генератора и дискриминатора независимо. Потери — бинарная кросс-энтропия с `from_logits=True`.

In [ ]:
g_optimizer=keras.optimizers.Adam(2e-4,beta_1=0.5)
d_optimizer=keras.optimizers.Adam(2e-4,beta_1=0.5)
bce=keras.losses.BinaryCrossentropy(from_logits=True)
def generator_loss(fake_logits):
    return bce(tf.ones_like(fake_logits),fake_logits)
def discriminator_loss(real_logits,fake_logits):
    return bce(tf.ones_like(real_logits),real_logits)+bce(tf.zeros_like(fake_logits),fake_logits)

## 5. Обучение дискриминатора

Берём реальные фотографии, создаём поддельные генератором и обновляем **только** дискриминатор. Это первая половина шага GAN.

In [ ]:
@tf.function
def train_discriminator(real):
    batch=tf.shape(real)[0]
    noise=tf.random.normal((batch,latent_dim))
    with tf.GradientTape() as tape:
        fake=generator(noise,training=True)
        real_logits=discriminator(real,training=True)
        fake_logits=discriminator(fake,training=True)
        loss=discriminator_loss(real_logits,fake_logits)
    grads=tape.gradient(loss,discriminator.trainable_variables)
    d_optimizer.apply_gradients(zip(grads,discriminator.trainable_variables))
    return loss

## 6. Обучение генератора

Создаём новый шум и обновляем **только** генератор так, чтобы дискриминатор принимал поддельные изображения за настоящие.

In [ ]:
@tf.function
def train_generator(batch):
    noise=tf.random.normal((batch,latent_dim))
    with tf.GradientTape() as tape:
        fake=generator(noise,training=True)
        fake_logits=discriminator(fake,training=False)
        loss=generator_loss(fake_logits)
    grads=tape.gradient(loss,generator.trainable_variables)
    g_optimizer.apply_gradients(zip(grads,generator.trainable_variables))
    return loss

## 7. Общий цикл обучения GAN

На каждом батче выполняются обе стадии; после каждой эпохи сохраняем изображения с фиксированным шумом. `EPOCHS=5` — быстрый учебный прогон, в методичке демонстрируется результат после 30 эпох.

In [ ]:
EPOCHS=5
fixed_noise=tf.random.normal((16,latent_dim),seed=42)
d_history=[];g_history=[]
def show_generated(epoch):
    imgs=generator(fixed_noise,training=False).numpy()
    fig,axs=plt.subplots(4,4,figsize=(6,6))
    for i,ax in enumerate(axs.flat):
        ax.imshow(np.clip(imgs[i],0,1));ax.axis("off")
    fig.suptitle(f"Эпоха {epoch}")
    plt.tight_layout();plt.show()
for epoch in range(1,EPOCHS+1):
    d_losses=[];g_losses=[]
    for real_batch in dataset:
        d_losses.append(float(train_discriminator(real_batch)))
        g_losses.append(float(train_generator(tf.shape(real_batch)[0])))
    d_history.append(float(np.mean(d_losses)))
    g_history.append(float(np.mean(g_losses)))
    print(f"Epoch {epoch}/{EPOCHS}: D={d_history[-1]:.4f}, G={g_history[-1]:.4f}")
    show_generated(epoch)

## 8. Результат работы нейронной сети

Визуализируем сгенерированные лица и графики потерь. Значения GAN loss сами по себе не являются метрикой визуального качества.

In [ ]:
show_generated(EPOCHS)
plt.plot(d_history,label="discriminator")
plt.plot(g_history,label="generator")
plt.xlabel("Эпоха");plt.ylabel("Loss");plt.legend();plt.show()

## Вывод

После запуска опишите, улучшается ли визуальное качество лиц, насколько устойчиво обучение и как влияют число эпох и объём выборки. Реальные изображения и значения потерь появятся только после выполнения.